In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from linearmodels import PanelOLS, RandomEffects, PooledOLS
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import chi2, jarque_bera

# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────
df = pd.read_csv("Data - raw/prepared_data/panel.csv")

hist = df[df["demand_oth_TWh"].notna()].copy()
proj = df[df["demand_oth_TWh"].isna()].copy()

# ── 2. LOG TRANSFORMATION ─────────────────────────────────────────────────────
for frame in [hist, proj]:
    frame["ln_demand_oth"] = np.log(frame["demand_oth_TWh"])
    frame["ln_pop"]        = np.log(frame["population_Mio"])
    frame["ln_gdp_pc"]     = np.log(frame["gdp_per_capita_USD"])

# ── 3. STATIONARITY — ADF on EU aggregate ─────────────────────────────────────
print("=" * 55)
print("ADF STATIONARITY TEST (EU aggregate time series)")
print("=" * 55)
eu_agg = hist.groupby("year")[["ln_demand_oth", "ln_pop", "ln_gdp_pc"]].mean()
for col in ["ln_demand_oth", "ln_pop", "ln_gdp_pc"]:
    stat, p, *_ = adfuller(eu_agg[col], autolag="AIC")
    print(f"  {col:<20}  stat={stat:7.3f}  p={p:.4f}  "
          f"{'STATIONARY' if p < 0.05 else 'NON-STATIONARY'}")

# ── 4. VIF CHECK ──────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("VIF (multicollinearity)")
print("=" * 55)
X_vif = hist[["ln_pop", "ln_gdp_pc"]].dropna()
for i, col in enumerate(X_vif.columns):
    print(f"  {col:<20}  VIF = {variance_inflation_factor(X_vif.values, i):.2f}")

# ── 5. SET PANEL INDEX ────────────────────────────────────────────────────────
panel = hist.set_index(["country", "year"]).sort_index()

# ── 6. TRAIN / VALIDATION SPLIT ───────────────────────────────────────────────
TRAIN_END = 2015
VAL_START = 2016

train = panel[panel.index.get_level_values("year") <= TRAIN_END]
val   = panel[panel.index.get_level_values("year") >= VAL_START]

print(f"\nTrain: {hist.year.min()}–{TRAIN_END}  ({len(train)} obs)")
print(f"Val  : {VAL_START}–{hist.year.max()}  ({len(val)} obs)")

# ── 7. ESTIMATE THREE MODELS ──────────────────────────────────────────────────
formula = "ln_demand_oth ~ ln_pop + ln_gdp_pc"

res_pooled = PooledOLS.from_formula(formula, data=train).fit(
    cov_type="clustered", cluster_entity=True
)
res_fe = PanelOLS.from_formula(formula + " + EntityEffects", data=train).fit(
    cov_type="clustered", cluster_entity=True
)
res_re = RandomEffects.from_formula(formula, data=train).fit(cov_type="robust")

print("\n" + "=" * 55)
print("POOLED OLS")
print("=" * 55)
print(res_pooled.summary.tables[1])

print("\n" + "=" * 55)
print("FIXED EFFECTS (PanelOLS)")
print("=" * 55)
print(res_fe.summary.tables[1])
print(f"  R² within : {res_fe.rsquared:.4f}")
print(f"  R² overall: {res_fe.rsquared_overall:.4f}")

print("\n" + "=" * 55)
print("RANDOM EFFECTS")
print("=" * 55)
print(res_re.summary.tables[1])

# ── 8. HAUSMAN TEST ───────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("HAUSMAN TEST (Fixed vs. Random Effects)")
print("=" * 55)

b_fe   = res_fe.params
b_re   = res_re.params[b_fe.index]
v_diff = res_fe.cov.loc[b_fe.index, b_fe.index] - res_re.cov.loc[b_fe.index, b_fe.index]
H      = float((b_fe - b_re) @ np.linalg.inv(v_diff) @ (b_fe - b_re))
p_H    = 1 - chi2.cdf(H, df=len(b_fe))

print(f"  H statistic: {H:.4f}  |  p-value: {p_H:.4f}")
if p_H < 0.05:
    print("  → p < 0.05: use FIXED EFFECTS")
    best_res  = res_fe
    best_name = "Fixed Effects"
else:
    print("  → p >= 0.05: use RANDOM EFFECTS")
    best_res  = res_re
    best_name = "Random Effects"

# ── 9. RESIDUAL DIAGNOSTICS ───────────────────────────────────────────────────
print("\n" + "=" * 55)
print(f"DIAGNOSTICS — {best_name}")
print("=" * 55)

resids = best_res.resids
dw     = durbin_watson(resids.values)
jb, pj = jarque_bera(resids.values)

print(f"  Residual mean  : {resids.mean():.6f}  (should be ~0)")
print(f"  Durbin-Watson  : {dw:.4f}  "
      f"({'OK' if 1.5 < dw < 2.5 else 'WARNING: autocorrelation likely'})")
print(f"  Jarque-Bera    : stat={jb:.3f}, p={pj:.4f}  "
      f"({'normal residuals' if pj > 0.05 else 'non-normal residuals'})")

# ── 10. OUT-OF-SAMPLE MAPE ────────────────────────────────────────────────────
print("\n" + "=" * 55)
print(f"BACKTESTING MAPE ({VAL_START}–{hist.year.max()})")
print("=" * 55)

features     = ["ln_pop", "ln_gdp_pc"]
val_pred_log = val[features] @ best_res.params[features] + resids.mean()
val_pred_twh = np.exp(val_pred_log)
val_act_twh  = np.exp(val["ln_demand_oth"])

ape          = (val_pred_twh - val_act_twh).abs() / val_act_twh * 100
mape_country = ape.reset_index().groupby("country")[0].mean().sort_values()

print(f"\n  Overall MAPE: {ape.mean():.2f}%")
print("\n  By country:")
print(mape_country.round(2).to_string())

# ── 11. EXTRACT RESIDUALS FOR ML LAYER ────────────────────────────────────────
panel["ols_residual"] = best_res.resids
panel["ols_fitted"]   = best_res.fitted_values
panel["resid_lag1"]   = panel.groupby(level="country")["ols_residual"].shift(1)
panel["resid_lag2"]   = panel.groupby(level="country")["ols_residual"].shift(2)

panel.reset_index().to_csv("panel_with_residuals.csv", index=False)
print("\n  Saved: panel_with_residuals.csv  -> use as XGBoost input")

ADF STATIONARITY TEST (EU aggregate time series)
  ln_demand_oth         stat= -1.963  p=0.3032  NON-STATIONARY
  ln_pop                stat=  1.163  p=0.9957  NON-STATIONARY
  ln_gdp_pc             stat= -0.711  p=0.8440  NON-STATIONARY

VIF (multicollinearity)
  ln_pop                VIF = 4.12
  ln_gdp_pc             VIF = 4.12

Train: 1990–2015  (676 obs)
Val  : 2016–2024  (234 obs)

POOLED OLS
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
ln_pop         0.8930     0.0660     13.520     0.0000      0.7633      1.0226
ln_gdp_pc      0.1230     0.0166     7.4016     0.0000      0.0904      0.1556

FIXED EFFECTS (PanelOLS)
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------